# Model Training & Evaluation

Implements Design Doc §5.3-§5.5 and [IMPLEMENTATION_PLAN.md](../IMPLEMENTATION_PLAN.md) Phase 4: scaffold split, XGBoost-on-ECFP vs. MLP-on-ChemBERTa, evaluated with RMSE/R²/Spearman.

This notebook is being built incrementally, one plan step at a time. **This pass covers only step 1: the scaffold split.** Model training (steps 2-3) and evaluation (steps 4-7) come in later passes.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append("../src")
from data_utils import scaffold_split

PROCESSED_DIR = Path("../data/processed")

df = pd.read_csv(PROCESSED_DIR / "kit_bioactivity_clean.csv")
print("Loaded:", df.shape)
df.head()

Loaded: (5565, 10)


,molecule_chembl_id,kit_variant,canonical_smiles,p_value,censored,censored_direction,n_measurements,p_value_std,n_documents,standard_types
0,CHEMBL10,D816V,C[S+]([O-])c1ccc(-c2nc(-c3ccc(F)cc3)c(-c3ccncc...,5.000000,True,>,3.0,NaN,3.0,Kd
1,CHEMBL10,WT,C[S+]([O-])c1ccc(-c2nc(-c3ccc(F)cc3)c(-c3ccncc...,5.000000,True,>,13.0,NaN,4.0,"Kd,Ki"
2,CHEMBL101253,D816V,Clc1ccc(Nc2nnc(Cc3ccncc3)c3ccccc23)cc1,5.000000,True,>,3.0,NaN,3.0,Kd
3,CHEMBL101253,WT,Clc1ccc(Nc2nnc(Cc3ccncc3)c3ccccc23)cc1,6.677781,False,NaN,17.0,1.308074,7.0,"IC50,Kd,Ki"
4,CHEMBL101683,WT,O=C(Nc1ccc(Cl)cc1)c1ccccc1NCc1ccncc1,6.619789,False,NaN,1.0,NaN,1.0,IC50


## 1. Scaffold split (train/test)

Design Doc §5.4: use a **scaffold split**, not a random split — grouping compounds by Bemis-Murcko scaffold before splitting, since random splits let near-identical analogues leak across train/test and overestimate generalization.

`scaffold_split` (in [`src/data_utils.py`](../src/data_utils.py)) groups row indices by scaffold, then assigns whole scaffold groups — largest first — to train until an 80% target is hit, with the remainder (smaller, rarer-scaffold groups) going to test. WT/D816V rows of the same compound share identical SMILES and therefore identical scaffolds, so they always land on the same side of the split; no special-casing needed.

In [2]:
train_idx, test_idx = scaffold_split(df["canonical_smiles"].tolist(), frac_train=0.8, seed=0)

print(f"Train: {len(train_idx)} rows ({len(train_idx) / len(df):.1%})")
print(f"Test:  {len(test_idx)} rows ({len(test_idx) / len(df):.1%})")

train_df = df.iloc[train_idx]
test_df = df.iloc[test_idx]

Train: 4452 rows (80.0%)
Test:  1113 rows (20.0%)


### Verify no scaffold leakage between train and test

The whole point of a scaffold split is that no scaffold appears on both sides — confirm that directly rather than trusting the split logic didn't have an off-by-one.

In [3]:
from data_utils import bemis_murcko_scaffold

train_scaffolds = set(train_df["canonical_smiles"].map(bemis_murcko_scaffold))
test_scaffolds = set(test_df["canonical_smiles"].map(bemis_murcko_scaffold))

overlap = train_scaffolds & test_scaffolds
print(f"Unique scaffolds — train: {len(train_scaffolds)}, test: {len(test_scaffolds)}")
print(f"Scaffolds appearing in both: {len(overlap)}")
assert not overlap, "Scaffold leakage between train and test!"

# Sanity-check the WT/D816V co-location invariant: every compound's rows should
# be entirely in train or entirely in test, never split across both.
split_side = pd.Series("train", index=df.index)
split_side.iloc[test_idx] = "test"
sides_per_compound = df.groupby("molecule_chembl_id").apply(
    lambda g: split_side.loc[g.index].nunique(), include_groups=False
)
straddling = sides_per_compound[sides_per_compound > 1]
print(f"Compounds with rows split across train AND test: {len(straddling)}")
assert straddling.empty, "A compound's WT/D816V rows ended up on different sides of the split!"


Unique scaffolds — train: 814, test: 1113
Scaffolds appearing in both: 0
Compounds with rows split across train AND test: 0


### Summary

- 4,452 rows train / 1,113 rows test (80.0%/20.0% split), grouped by 814 train + 1,113 test Bemis-Murcko scaffolds rather than randomly.
- Zero scaffold overlap between train and test confirmed directly.
- Every compound's WT/D816V row pair confirmed to land entirely on one side of the split (never straddling), since they share identical scaffolds. In practice, all 925 compounds with both WT and D816V measurements happen to land in train under this split/seed — worth keeping in mind for Phase 5 (selectivity analysis), which relies on paired WT/D816V compounds.
- Split is seeded and deterministic (`seed=0`); a different seed changes which scaffold groups land in test while preserving the ~80/20 split size and the zero-leakage guarantee.
- Next (not yet done in this pass): load the cached ECFP/ChemBERTa features (`03_featurization.ipynb`), align them to `train_idx`/`test_idx`, and train the XGBoost and MLP models (Phase 4 steps 2-3).